# QLoRA fine-tune + eval — Kaggle GPU notebook

Runs the **only GPU-dependent step** of the thesis (Phase-2 B4): fine-tune Qwen2.5-Coder on the hybrid-linearized slices and report **F1 / PR-AUC** per variant. Everything else (corpus generation, ablation slicing, robustness, external comparison) runs on your CPU VM.

## Before you run
1. **Settings → Accelerator = GPU (T4 x2 or P100)**, **Internet = On**.
2. `build/ft_dataset.jsonl.gz` in the repo must contain **train + val + test** rows. Generate train/val on your CPU VM first (see `build/qlora_cloud.md` / `NEXT_STEPS.md`) and push. Until then the committed corpus is test-only and the training cell stops with a clear message.
3. Private repo: **Add-ons → Secrets →** add your GitHub PAT as a secret named `GITHUB_TOKEN`.

In [ ]:
# 1) Confirm the GPU
!nvidia-smi -L
import torch
print('cuda', torch.cuda.is_available(),
      '| bf16', torch.cuda.is_available() and torch.cuda.is_bf16_supported())

In [ ]:
# 2) Clone the repo (uses the GITHUB_TOKEN secret if the repo is private)
import os
REPO = 'KeyboardRules/Master-Thesis'
tok = ''
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    pass  # public repo, or secret not set
auth = tok + '@' if tok else ''
url = 'https://' + auth + 'github.com/' + REPO + '.git'
%cd /kaggle/working
!rm -rf Master-Thesis
!git clone --depth 1 $url
%cd Master-Thesis

In [ ]:
# 3) Install the training/eval dependencies
!pip -q install 'transformers>=4.44' peft bitsandbytes datasets accelerate scikit-learn

In [ ]:
# 4) Unpack the corpus and check that train rows exist BEFORE spending GPU time
!gunzip -kf build/ft_dataset.jsonl.gz
import json, collections
rows = [json.loads(l) for l in open('build/ft_dataset.jsonl')]
by_split = collections.Counter(r['split'] for r in rows)
by_variant = collections.Counter(r['variant'] for r in rows)
print('rows by split  :', dict(by_split))
print('rows by variant:', dict(by_variant))
assert by_split.get('train', 0) > 0, (
    'No train rows — the corpus is test-only. Generate train/val on the VM first '
    '(python build/run_phase2.py --split train/--split val --max-php-kb N), gzip, push, re-pull.')

In [ ]:
# 5) Fine-tune + evaluate. 1.5B suits the small corpus and fits a free T4;
#    the script auto-uses fp16 on T4/P100 and bf16 on Ampere+.
MODEL = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
!python build/qlora_train_eval.py --data build/ft_dataset.jsonl --model $MODEL --output out --epochs 3

## Reading the result

Cell 5 prints an **F1 / PR-AUC** table by variant and CWE. **Acceptance:** `variant:cross-module` beats both `variant:intra-file` and `variant:no-slice`.

Reporting rules (`build/THREATS_TO_VALIDITY.md`): score on the **raw** file (no de-dup), quote **PR-AUC + positive rate** next to F1, keep the three variants paired.

The LoRA adapter is saved under `out/adapter/`. Re-score without retraining:
```
!python build/qlora_train_eval.py --data build/ft_dataset.jsonl --adapter out/adapter --eval_only --model $MODEL
```
Copy the printed table into the thesis (Evaluation chapter). To also try 7B, set `MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'` (tight on a free T4; better on a rented 24 GB GPU).